# Paperbunkr panel model: fine-tune

Fine-tunes guided view's panel detector (YOLO26-nano, `leoxs22/manga-panel-detector-yolo26n`) into a one-class **panel** model on your
corrected pages plus a Manga109-s sample, then exports ONNX at 1024 and 640 px for the app.
Design: `docs/superpowers/specs/2026-11-07-panel-model-retraining-design.md` (plan Step 9/10). AGPL-3.0, like the model.

**Before running (Kaggle notebook settings, right-hand panel):**
1. **Accelerator:** GPU (P100, or T4; one GPU is used).
2. **Internet:** on (to install ultralytics and download the base weights).
3. **Add Input:** your private dataset **paperbunkr-panels** (the zip from `make_kaggle_zip.py`).
4. **Manga109-s** comes from your private Hugging Face bucket `HF_BUCKET` (set below): in the notebook editor, Add-ons > Secrets,
   add `HF_TOKEN` (a **read** token from huggingface.co/settings/tokens, same account as the bucket) and switch it on for this notebook.
   Alternatives the notebook also accepts: an attached Kaggle dataset holding the Manga109-s zip, or access to the gated
   `hal-utokyo/Manga109-s` dataset with the same token.

**First run with `SMOKE = True`** (1 epoch, a few Manga109 pages, a few minutes) to check everything works. Then set it to `False` and run all.
Nothing here leaves your Kaggle account. Manga109-s images stay in `/tmp` and are not saved to the notebook output.

In [ ]:
SMOKE = True          # True: a quick check run. False: the real training run.
RUN_NAME = "batch1"   # "batch1" for the first model, "final" for the batch 1 + batch 2 model
SEED = 1

EPOCHS = 1 if SMOKE else 60
M109_PAGES = 40 if SMOKE else 1500
ULTRALYTICS = "8.4.117"   # pinned: 8.4.142 changed the end-to-end export the app relies on
HF_BUCKET = "Mrehis/Manga109-s-bucket"   # your private bucket holding Manga109-s ("" to skip it)

In [ ]:
!pip install -q "ultralytics=={ULTRALYTICS}" onnxruntime "huggingface_hub>=2.4"

In [ ]:
import glob, os, shutil, sys

def find(pattern):
    hits = sorted(glob.glob(pattern, recursive=True))
    return hits[0] if hits else None

own_train = find("/kaggle/input/**/images/train")
assert own_train, "Attach your private dataset 'paperbunkr-panels' (Add Input)."
OWN = os.path.dirname(os.path.dirname(own_train))
print("your pages:", OWN)
print(open(os.path.join(OWN, "SUMMARY.txt")).read())
sys.path.insert(0, OWN)
import manga109_to_yolo

In [ ]:
# Manga109-s: an attached Kaggle copy (zip or unpacked) first, then your bucket, then the gated dataset.
def hf_token():
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = token   # huggingface_hub reads it from here
    return token

src = find("/kaggle/input/**/Manga109s*.zip")
if src is None:
    ann = find("/kaggle/input/**/annotations/*.xml")
    src = "/".join(ann.split("/")[:4]) if ann else None
if src is None and HF_BUCKET:
    hf_token()
    from huggingface_hub import list_bucket_tree, download_bucket_files, sync_bucket
    files = [f for f in list_bucket_tree(HF_BUCKET, recursive=True) if f.type == "file"]
    print(f"bucket {HF_BUCKET}: {len(files)} files, {sum(f.size for f in files) / 1e9:.1f} GB")
    zips = [f for f in files if f.path.lower().endswith(".zip")]
    if zips:
        biggest = max(zips, key=lambda f: f.size)
        src = f"/tmp/m109src/{os.path.basename(biggest.path)}"
        download_bucket_files(HF_BUCKET, files=[(biggest, src)])
    else:
        sync_bucket(f"hf://buckets/{HF_BUCKET}", "/tmp/m109src")
        src = "/tmp/m109src"
if src is None:
    from huggingface_hub import hf_hub_download
    src = hf_hub_download(repo_id="hal-utokyo/Manga109-s", repo_type="dataset",
                          filename="Manga109s_released_2026_05_21.zip", token=hf_token(), local_dir="/tmp/hf")
print("Manga109-s from:", src)
shutil.rmtree("/tmp/m109", ignore_errors=True)
counts = manga109_to_yolo.convert(src, "/tmp/m109", M109_PAGES, 0.1, SEED)
print("Manga109-s pages:", counts)

In [ ]:
from huggingface_hub import hf_hub_download
from ultralytics import YOLO
BASE = hf_hub_download("leoxs22/manga-panel-detector-yolo26n", "manga_panel_detector_fp32.pt", local_dir="/tmp/base")

# One class, named like the base model's panel class ('frame'; its other class is 'text'). ultralytics' cls_remap
# (on by default) copies the head row by class name, so the name must match or the class head restarts from random
# weights (the log then says 'Transferred 696/708' with no 'Remapped 1/1 cls head rows' line). The app only reads class 0.
assert "frame" in YOLO(BASE).names.values(), f"base model classes changed: {YOLO(BASE).names}"

with open("/tmp/panels.yaml", "w") as f:
    f.write(f"""path: /
train:
  - {OWN}/images/train
  - /tmp/m109/images/train
val:
  - {OWN}/images/val
  - /tmp/m109/images/val
names:
  0: frame
""")
print(open("/tmp/panels.yaml").read())

In [ ]:
from ultralytics import YOLO

model = YOLO(BASE)
model.train(
    data="/tmp/panels.yaml", imgsz=1024, epochs=EPOCHS, patience=15, batch=16, device=0, seed=SEED, deterministic=True,
    # The base model's augmentation: no colour shift (black-and-white pages), no flips (reading order), no rotation.
    hsv_h=0.0, hsv_s=0.0, fliplr=0.0, mosaic=0.5, degrees=0.0,
    project="/kaggle/working/runs", name=RUN_NAME, exist_ok=True, workers=4,
)

In [ ]:
import onnxruntime as ort

RUN = f"/kaggle/working/runs/{RUN_NAME}"
OUT = f"/kaggle/working/out-{RUN_NAME}"
os.makedirs(OUT, exist_ok=True)
for size in (1024, 640):
    exported = YOLO(f"{RUN}/weights/best.pt").export(format="onnx", imgsz=size, opset=17, end2end=True, nms=False, max_det=300)
    target = f"{OUT}/panel-{RUN_NAME}-{size}.onnx"
    shutil.move(exported, target)
    session = ort.InferenceSession(target)
    shape_in, shape_out = session.get_inputs()[0].shape, session.get_outputs()[0].shape
    print(size, "in", shape_in, "out", shape_out)
    assert list(shape_out) == [1, 300, 6], f"output {shape_out}: the app needs (1, 300, 6)"
for name in ("results.csv", "args.yaml"):
    shutil.copy(f"{RUN}/{name}", OUT)
shutil.copy(f"{RUN}/weights/best.pt", f"{OUT}/best-{RUN_NAME}.pt")
archive = shutil.make_archive(f"/kaggle/working/panel-model-{RUN_NAME}", "zip", OUT)
print("download:", archive, f"({os.path.getsize(archive) / 1e6:.0f} MB)")

**Next:** download `panel-model-<RUN_NAME>.zip` from the notebook's Output panel and unzip it into
`C:\Users\DeeDee\PaperBunkr-panel-data\models\`, then tell Claude. The harness scores both ONNX files on the test pages.

After the **final** run, delete the private dataset `paperbunkr-panels` (and any Manga109-s copy) from Kaggle.